# RDDT V5 amyloidosis pipeline

Runs the version 4 confirmation check first, then the version 5 suspicion check. Confirmation is the same E85 allow-list as version 4, including AA, AL, and ATTR. Every confirmed patient is written under `confirmed/`. Suspicion profiles are separate: ATTR under `detected/`, AL under `al_detected/`, AA under `aa_detected/`. Highest and high suspicion each get a profile. Moderate suspicion is counted and does not get a profile. Highest-suspicion profiles are written first. The profile text for AL and AA uses the same expected-versus-matched wording as ATTR.

Labs, census, surgical history, and medications do not fire atoms. They are read only for a patient who is confirmed or who passes suspicion, and they are stored on that patient's profile.

The scan scores every screened patient in batches of 200. Census is copied once in Snowflake for those patients. Each batch is scored once, its verdicts are saved, and profiles for the confirmed and highest and high patients in that batch are written from that same pass. Chart tables are read once for the group, not once per patient. Counts are printed after every batch, with an estimate of the hours left.

If the notebook stops, run the continue cell below. Patients who already have a verdict are not scored again, and patients who still need a profile get one before any new scoring. A new Snowflake session reads the permanent TEMP_V5 tables. It does not rebuild them and it does not start the patient scan over. One damaged checkpoint file is skipped. The other saved files stay. A patient whose screening rows are too large to load is saved as a failed score after one stop, and the scan continues.

The notebook uses the active Snowflake session. It creates only `AMY_V5_` temporary tables, with `CREATE TEMPORARY TABLE` and `INSERT`. It does not create a permanent table, transient table, view, or stage. It does not run `UPDATE`, `DELETE`, `DROP`, `ALTER`, `TRUNCATE`, `MERGE`, or `CREATE OR REPLACE`. Temporary tables disappear when the session ends. A statement that is not a read or a temporary-table create/insert is rejected before Snowflake sees it.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print({"execution": "V5_ICD_DATED_CLAIMS", "connection": "get_active_session"}, flush=True)

In [ ]:
from pathlib import Path
import sys

package_candidates = [Path.cwd() / "v5", Path.cwd(), Path.cwd().parent / "v5"]
package_dir = next((p.resolve() for p in package_candidates if (p / "__init__.py").exists() and (p / "config").is_dir()), None)
if package_dir is None:
    raise RuntimeError("Run this notebook from the v5 folder or its parent directory.")
sys.path.insert(0, str(package_dir.parent))

import v5.tools.run_warehouse_pipeline as warehouse_pipeline
from v5.tools.run_warehouse_pipeline import run_icd_dated_claims_warehouse

SOURCE_NAMESPACE = "UHTX_RDDT_CLINICAL_DEV.PUBLIC"  # DATABASE.SCHEMA
SCREENING_CUTOFF = "2026-09-22"
OUTPUT_DIR = Path.cwd() / "v5_warehouse_exports"
FRESH_START = False  # True deletes saved verdicts and profiles in OUTPUT_DIR

file_is_current = bool(getattr(warehouse_pipeline, "CHECKPOINT_PROFILES", False))
if not file_is_current:
    raise RuntimeError(
        "The checkpointing run_warehouse_pipeline.py is not loaded. Scoring was not started. "
        "Replace v5/tools/run_warehouse_pipeline.py on the Snowflake filesystem, then run this cell again."
    )

scan_summary = run_icd_dated_claims_warehouse(
    session,
    namespace=SOURCE_NAMESPACE,
    screening_cutoff=SCREENING_CUTOFF,
    output_dir=OUTPUT_DIR,
    score_batch_size=200,
    patient_limit=None,
    resume=True,
    fresh_start=FRESH_START,
)
scan_summary

## Continue after a stop

Run the cell below when the scan cell stops, the kernel restarts, or the Snowflake session ends. It reloads the pipeline file, keeps every saved verdict and profile, writes any missing profiles in batches, then scores the remaining patients in batches of 200. A batch that ran out of memory is cut in half on the next start. A patient who still stops the notebook alone is saved as failed, and the scan moves on. Counts and profiles are both saved. `suspicious_amyloidosis` is highest, high, or moderate ATTR, AL, or AA. `highest_attr`, `highest_al`, and `highest_aa` are counted on their own, and so are the high counts. The log prints `eta_hours` after every batch.

In [ ]:
from pathlib import Path
import importlib

import v5.tools.run_warehouse_pipeline as warehouse_pipeline

importlib.reload(warehouse_pipeline)
run_icd_dated_claims_warehouse = warehouse_pipeline.run_icd_dated_claims_warehouse
file_is_current = bool(getattr(warehouse_pipeline, "CHECKPOINT_PROFILES", False))
print({
    "pipeline_file": warehouse_pipeline.__file__,
    "checkpoint_profiles": file_is_current,
}, flush=True)
if not file_is_current:
    raise RuntimeError(
        "The checkpointing run_warehouse_pipeline.py is not loaded. Scoring was not started. "
        "Replace v5/tools/run_warehouse_pipeline.py on the Snowflake filesystem, then run this cell again."
    )

SOURCE_NAMESPACE = "UHTX_RDDT_CLINICAL_DEV.PUBLIC"  # DATABASE.SCHEMA
SCREENING_CUTOFF = "2026-09-22"
OUTPUT_DIR = Path.cwd() / "v5_warehouse_exports"

scan_summary = run_icd_dated_claims_warehouse(
    session,
    namespace=SOURCE_NAMESPACE,
    screening_cutoff=SCREENING_CUTOFF,
    output_dir=OUTPUT_DIR,
    score_batch_size=200,
    patient_limit=None,
    resume=True,
    fresh_start=False,
)
scan_summary